In [1]:
import numpy as np
import tensorflow as tf
import string

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

In [2]:
# Load IMDB word index

max_features = 10000
max_len = 500

In [3]:
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

In [ ]:
# Load Trained Model

model = load_model("sentiment_model.h5")
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (32, 500, 128)         │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (32, 128)              │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (32, 1)                │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,411,715 (5.39 MB)

 Trainable params: 1,411,713 (5.39 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [5]:
# Helper Function: Decode Review

def decode_review(encoded_review):
    return " ".join([reverse_word_index.get(i - 3, "?") for i in encoded_review])

In [ ]:
# CORRECT Preprocessing Function

def preprocess_text(text):
    # Lowercase
    text = text.lower()
    # Remove punctuation (CRITICAL FIX)
    text = text.translate(str.maketrans('', '', string.punctuation))
    words = text.split()
    encoded_review = []
    for word in words:
        index = word_index.get(word)
        # Match training vocab limit
        if index is not None and index < max_features:
            encoded_review.append(index + 3)
        else:
            encoded_review.append(2)  # OOV token

    padded_review = sequence.pad_sequences([encoded_review], maxlen=max_len)

    return padded_review

In [ ]:
# Prediction Function

def predict_sentiment(review):
    preprocessed_input = preprocess_text(review)
    prediction = model.predict(preprocessed_input, verbose=0)
    score = float(prediction[0][0])
    sentiment = "Positive" if score >= 0.5 else "Negative"
    return sentiment, score

In [ ]:
# Example Prediction

example_review = "This movie was worst and boring. I hated it."

sentiment, score = predict_sentiment(example_review)
print("Review:", example_review)
print("Sentiment:", sentiment)
print("Prediction Score:", round(score, 4))

Review: This movie was worst and boring. I hated it.
Sentiment: Negative
Prediction Score: 0.0937
